In [31]:
!pip install selenium-wire webdriver-manager

Defaulting to user installation because normal site-packages is not writeable
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
Using cached PySocks-1.7.1-py3-none-any.whl (16 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.7 MB 2.9 MB/s eta 0:00:04
   ---- ----------------------------------- 1.0/9.7 MB 2.4 MB/s eta 0:00:04
   ------ --------------------------------- 1.6/9.7 MB 2.4 MB/s eta 0:00:04
   ------- -------------------------------- 1.8/9.7 MB 2.2 MB/s eta 0:00:04
   --------- ------------------------------ 2.4/9.7 MB 2.1 MB/s eta 0:00:04
   ---------- ----------------------------- 2.6/9.7 MB 2.1 MB/s eta 0:00:04
   --------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [40]:
import csv
import re
import os
import time
import random
from datetime import datetime
import traceback

# Menggunakan Selenium Standar murni (Bebas ketergantungan pyOpenSSL/Mitmproxy)
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ─────────────────────────────────────────────
# KONFIGURASI UTAMA
# ─────────────────────────────────────────────

DAFTAR_URL_VIDEO = [
    "https://vt.tiktok.com/ZSxmNoQ4Q/",
    "https://vt.tiktok.com/ZSxmF9oqJ/",
    "https://vt.tiktok.com/ZSxmFVNY4/",
    "https://vt.tiktok.com/ZSxmFTtAu/",
    "https://vt.tiktok.com/ZSxmYkrqy/",
    "https://vt.tiktok.com/ZSxm2QSkk/",
    "https://vt.tiktok.com/ZSxmj5QyF/",
    "https://vt.tiktok.com/ZSxmjce15/",
]

NAMA_FILE_OUTPUT = "dataset_tiktok.csv"
TARGET_JUMLAH = 23000

JEDA_MIN = 4
JEDA_MAX = 8

# ─────────────────────────────────────────────
# KAMUS KEYWORD & LABELING
# ─────────────────────────────────────────────

KAMUS_KATEGORI = {
    "Kata Kasar": ["anjing", "anjir", "bangsat", "brengsek", "bajingan", "goblok", "tolol", "idiot", "bego", "bodoh", "tai", "sialan", "asu", "cok", "kontol", "memek", "pantek"],
    "Penipuan": ["transfer sekarang", "kirim uang", "mama minta pulsa", "menang hadiah", "klik link", "verifikasi akun", "nomor rekening", "anda terpilih", "bonus cashback", "undian berhadiah"],
    "Judi Online": ["slot", "gacor", "maxwin", "jackpot", "scatter", "pragmatic", "pg soft", "mahjong", "rtp tinggi", "bocoran slot", "deposit", "withdraw", "wd", "spin", "bet"],
    "Pelecehan": ["jelek banget", "gemuk", "gendut", "kurus kering", "item", "pesek", "pendek", "cebol", "bau", "miskin", "melarat", "gak laku", "mati aja", "sampah masyarakat"],
    "Transaksi": ["transfer", "bayar", "pembayaran", "rekening", "bca", "bri", "mandiri", "bni", "dana", "ovo", "gopay", "shopeepay", "qris", "tagihan", "cicilan", "lunas"],
}

LABEL_MAP = {
    "Kata Kasar": "Berisiko", "Penipuan": "Berisiko", "Judi Online": "Berisiko",
    "Pelecehan": "Berisiko", "Transaksi": "Tidak Berisiko", "Netral": "Tidak Berisiko"
}

def klasifikasi_pesan(teks: str) -> tuple[str, str]:
    teks_lower = teks.lower()
    for kategori in ["Judi Online", "Penipuan", "Kata Kasar", "Pelecehan", "Transaksi"]:
        for kata in KAMUS_KATEGORI[kategori]:
            if re.search(re.escape(kata), teks_lower):
                return kategori, LABEL_MAP[kategori]
    return "Netral", "Tidak Berisiko"

# ─────────────────────────────────────────────
# CORE ENGINE - SELENIUM DOM TEXT SCRAPER
# ─────────────────────────────────────────────

def scrape_komentar_tiktok_dom(url_video: str, max_komentar: int = 500) -> list[str]:
    komentar_set = set()
    
    options = webdriver.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--start-maximized")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36")
    
    driver = None
    try:
        print("  🚀 Meluncurkan browser Chrome murni...")
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        
        print(f"  🌐 Membuka Target: {url_video}")
        driver.get(url_video)
        
        print("  ⏳ Menunggu 15 detik awal... SILAKAN SELESAIKAN CAPTCHA JIKA MUNCUL!")
        time.sleep(15)
        
        # Coba tutup modal login jika menghalangi layar
        try:
            close_btn = driver.find_element(By.CSS_SELECTOR, '[data-e2e="modal-close-inner-button"]')
            close_btn.click()
            print("  📥 Berhasil menutup modal popup otomatis.")
        except Exception:
            pass

        print("  ⏳ Memulai pemindaian elemen dan scroll otomatis...")
        scroll_attempts = 0
        max_attempts = 40
        last_count = 0
        
        while len(komentar_set) < max_komentar and scroll_attempts < max_attempts:
            # Cari seluruh elemen teks komentar menggunakan selector data-e2e bawaan TikTok
            elemen_komentar = driver.find_elements(By.CSS_SELECTOR, '[data-e2e="comment-level-1-text"]')
            
            for el in elemen_komentar:
                try:
                    teks = el.text.strip()
                    if teks and len(teks) > 1:
                        komentar_set.add(teks)
                except Exception:
                    continue
            
            current_count = len(komentar_set)
            if current_count > last_count:
                print(f"  📝 [DOM Scraping] Berhasil menarik {current_count} komentar unik sejauh ini.")
                last_count = current_count
                scroll_attempts = 0  # reset batas macet jika ada data baru masuk
            
            # Simulasi scroll ke bawah halaman untuk memicu lazy-load
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(random.uniform(2.0, 3.5))
            
            # Variasi gerakan scroll naik sedikit lalu banting ke bawah lagi
            driver.execute_script("window.scrollBy(0, -150);")
            time.sleep(0.3)
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            
            scroll_attempts += 1
            
            # Jika macet tidak bertambah dalam beberapa scroll, ingatkan pengguna
            if scroll_attempts == 6 and current_count == 0:
                print("  ⚠️ Deteksi web statis. Tolong taruh kursor dan SCROLL manual kolom komentarnya!")
                time.sleep(8)
                
        driver.quit()
        return list(komentar_set)
        
    except Exception as e:
        print("  ⚠️ Kendala fatal sistem scraping:")
        traceback.print_exc()
        if driver:
            driver.quit()
        return []

# ─────────────────────────────────────────────
# MANAJEMEN PENYIMPANAN DATA
# ─────────────────────────────────────────────

def simpan_ke_csv(data, nama_file, mode="w"):
    fieldnames = ["id", "pesan", "kategori", "label"]
    tulis_header = (mode == "w") or (not os.path.exists(nama_file))
    with open(nama_file, mode=mode, newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if tulis_header: 
            writer.writeheader()
        writer.writerows(data)

# ─────────────────────────────────────────────
# RUNNER UTAMA
# ─────────────────────────────────────────────

def main_runner():
    print("=" * 60)
    print("  SELENIUM DOM SCRAPER - DATASET TIKTOK (CLEAN EDITION)")
    print(f"  Target: {TARGET_JUMLAH:,} komentar | File: {NAMA_FILE_OUTPUT}")
    print(f"  Waktu mulai: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)
    
    id_counter = 1
    if os.path.exists(NAMA_FILE_OUTPUT):
        try:
            with open(NAMA_FILE_OUTPUT, "r", encoding="utf-8-sig") as f:
                rows = list(csv.DictReader(f))
                if rows: 
                    id_counter = int(rows[-1]["id"]) + 1
        except Exception: 
            pass
        
    total_terkumpul = id_counter - 1
    mode_file = "a" if total_terkumpul > 0 else "w"
    
    if total_terkumpul > 0:
        print(f"📂 Melanjutkan dataset dari baris ke-{id_counter}")

    for i, url in enumerate(DAFTAR_URL_VIDEO, 1):
        if total_terkumpul >= TARGET_JUMLAH:
            print("\n🎯 Target total komentar terpenuhi!")
            break
            
        sisa_target = TARGET_JUMLAH - total_terkumpul
        max_per_video = min(500, sisa_target)
        
        print(f"\n[{i}/{len(DAFTAR_URL_VIDEO)}] Memproses video...")
        komentar_mentah = scrape_komentar_tiktok_dom(url, max_per_video)
        
        if not komentar_mentah:
            print("  ⏩ Lewati video ini karena data kosong atau terjadi kendala.")
            continue
            
        batch_data = []
        for teks in komentar_mentah:
            kategori, label = klasifikasi_pesan(teks)
            batch_data.append({"id": id_counter, "pesan": teks, "kategori": kategori, "label": label})
            id_counter += 1
            
        simpan_ke_csv(batch_data, NAMA_FILE_OUTPUT, mode=mode_file)
        mode_file = "a"
        total_terkumpul += len(batch_data)
        
        print(f"  📊 Progress: +{len(batch_data)} data | Total Akumulasi: {total_terkumpul:,}/{TARGET_JUMLAH:,}")
        
        if i < len(DAFTAR_URL_VIDEO):
            jeda = random.uniform(JEDA_MIN, JEDA_MAX)
            print(f"  ⏳ Jeda bot {jeda:.1f} detik...")
            time.sleep(jeda)

    print("\n" + "=" * 60 + "\n  PROSES SELESAI DAN DATASET AMAN!\n" + "=" * 60)

main_runner()

  SELENIUM DOM SCRAPER - DATASET TIKTOK (CLEAN EDITION)
  Target: 23,000 komentar | File: dataset_tiktok.csv
  Waktu mulai: 2026-05-24 18:02:09

[1/8] Memproses video...
  🚀 Meluncurkan browser Chrome murni...


  🌐 Membuka Target: https://vt.tiktok.com/ZSxmNoQ4Q/
  ⏳ Menunggu 15 detik awal... SILAKAN SELESAIKAN CAPTCHA JIKA MUNCUL!
  ⏳ Memulai pemindaian elemen dan scroll otomatis...
  ⚠️ Deteksi web statis. Tolong taruh kursor dan SCROLL manual kolom komentarnya!
  ⏩ Lewati video ini karena data kosong atau terjadi kendala.

[2/8] Memproses video...
  🚀 Meluncurkan browser Chrome murni...
  🌐 Membuka Target: https://vt.tiktok.com/ZSxmF9oqJ/
  ⏳ Menunggu 15 detik awal... SILAKAN SELESAIKAN CAPTCHA JIKA MUNCUL!
  ⏳ Memulai pemindaian elemen dan scroll otomatis...
  ⚠️ Deteksi web statis. Tolong taruh kursor dan SCROLL manual kolom komentarnya!
  ⏩ Lewati video ini karena data kosong atau terjadi kendala.

[3/8] Memproses video...
  🚀 Meluncurkan browser Chrome murni...
  🌐 Membuka Target: https://vt.tiktok.com/ZSxmFVNY4/
  ⏳ Menunggu 15 detik awal... SILAKAN SELESAIKAN CAPTCHA JIKA MUNCUL!
  ⏳ Memulai pemindaian elemen dan scroll otomatis...
  ⚠️ Kendala fatal sistem scraping:


Traceback (most recent call last):
  File "C:\Users\Dimas\AppData\Local\Temp\ipykernel_26504\465596878.py", line 101, in scrape_komentar_tiktok_dom
    elemen_komentar = driver.find_elements(By.CSS_SELECTOR, '[data-e2e="comment-level-1-text"]')
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dimas\AppData\Local\Programs\Python\Python312\Lib\site-packages\selenium\webdriver\remote\webdriver.py", line 788, in find_elements
    return self.execute(Command.FIND_ELEMENTS, {"using": by, "value": value})["value"] or []
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dimas\AppData\Local\Programs\Python\Python312\Lib\site-packages\selenium\webdriver\remote\webdriver.py", line 380, in execute
    self.error_handler.check_response(response)
  File "c:\Users\Dimas\AppData\Local\Programs\Python\Python312\Lib\site-packages\selenium\webdriver\remote\errorhandler.py", line 229, in check_res

  ⏩ Lewati video ini karena data kosong atau terjadi kendala.

[4/8] Memproses video...
  🚀 Meluncurkan browser Chrome murni...
  🌐 Membuka Target: https://vt.tiktok.com/ZSxmFTtAu/
  ⏳ Menunggu 15 detik awal... SILAKAN SELESAIKAN CAPTCHA JIKA MUNCUL!
  ⏳ Memulai pemindaian elemen dan scroll otomatis...
  ⚠️ Kendala fatal sistem scraping:


Traceback (most recent call last):
  File "C:\Users\Dimas\AppData\Local\Temp\ipykernel_26504\465596878.py", line 101, in scrape_komentar_tiktok_dom
    elemen_komentar = driver.find_elements(By.CSS_SELECTOR, '[data-e2e="comment-level-1-text"]')
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dimas\AppData\Local\Programs\Python\Python312\Lib\site-packages\selenium\webdriver\remote\webdriver.py", line 788, in find_elements
    return self.execute(Command.FIND_ELEMENTS, {"using": by, "value": value})["value"] or []
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dimas\AppData\Local\Programs\Python\Python312\Lib\site-packages\selenium\webdriver\remote\webdriver.py", line 380, in execute
    self.error_handler.check_response(response)
  File "c:\Users\Dimas\AppData\Local\Programs\Python\Python312\Lib\site-packages\selenium\webdriver\remote\errorhandler.py", line 229, in check_res

...

  ⏩ Lewati video ini karena data kosong atau terjadi kendala.

[7/8] Memproses video...
  🚀 Meluncurkan browser Chrome murni...
  🌐 Membuka Target: https://vt.tiktok.com/ZSxmj5QyF/
  ⏳ Menunggu 15 detik awal... SILAKAN SELESAIKAN CAPTCHA JIKA MUNCUL!
  ⏳ Memulai pemindaian elemen dan scroll otomatis...
  ⚠️ Kendala fatal sistem scraping:


Traceback (most recent call last):
  File "C:\Users\Dimas\AppData\Local\Temp\ipykernel_26504\465596878.py", line 101, in scrape_komentar_tiktok_dom
    elemen_komentar = driver.find_elements(By.CSS_SELECTOR, '[data-e2e="comment-level-1-text"]')
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dimas\AppData\Local\Programs\Python\Python312\Lib\site-packages\selenium\webdriver\remote\webdriver.py", line 788, in find_elements
    return self.execute(Command.FIND_ELEMENTS, {"using": by, "value": value})["value"] or []
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dimas\AppData\Local\Programs\Python\Python312\Lib\site-packages\selenium\webdriver\remote\webdriver.py", line 380, in execute
    self.error_handler.check_response(response)
  File "c:\Users\Dimas\AppData\Local\Programs\Python\Python312\Lib\site-packages\selenium\webdriver\remote\errorhandler.py", line 229, in check_res

  ⏩ Lewati video ini karena data kosong atau terjadi kendala.

[8/8] Memproses video...
  🚀 Meluncurkan browser Chrome murni...
  🌐 Membuka Target: https://vt.tiktok.com/ZSxmjce15/
  ⏳ Menunggu 15 detik awal... SILAKAN SELESAIKAN CAPTCHA JIKA MUNCUL!
  ⏳ Memulai pemindaian elemen dan scroll otomatis...
  ⚠️ Kendala fatal sistem scraping:


Traceback (most recent call last):
  File "C:\Users\Dimas\AppData\Local\Temp\ipykernel_26504\465596878.py", line 101, in scrape_komentar_tiktok_dom
    elemen_komentar = driver.find_elements(By.CSS_SELECTOR, '[data-e2e="comment-level-1-text"]')
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dimas\AppData\Local\Programs\Python\Python312\Lib\site-packages\selenium\webdriver\remote\webdriver.py", line 788, in find_elements
    return self.execute(Command.FIND_ELEMENTS, {"using": by, "value": value})["value"] or []
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dimas\AppData\Local\Programs\Python\Python312\Lib\site-packages\selenium\webdriver\remote\webdriver.py", line 380, in execute
    self.error_handler.check_response(response)
  File "c:\Users\Dimas\AppData\Local\Programs\Python\Python312\Lib\site-packages\selenium\webdriver\remote\errorhandler.py", line 229, in check_res

  ⏩ Lewati video ini karena data kosong atau terjadi kendala.

  PROSES SELESAI DAN DATASET AMAN!


In [2]:
DAFTAR_URL_VIDEO = [
    # Contoh URL — ganti dengan URL video TikTok asli
    "https://vt.tiktok.com/ZSxmNoQ4Q/",
    "https://vt.tiktok.com/ZSxmF9oqJ/",
    "https://vt.tiktok.com/ZSxmFVNY4/",
    "https://vt.tiktok.com/ZSxmFTtAu/",
    "https://vt.tiktok.com/ZSxmYkrqy/",
    "https://vt.tiktok.com/ZSxm2QSkk/",
    "https://vt.tiktok.com/ZSxmj5QyF/",
    "https://vt.tiktok.com/ZSxmjce15/",
]
 
# Nama file output
NAMA_FILE_OUTPUT = "dataset_tiktok.csv"
 
# Target jumlah data (opsional, set None untuk ambil semua)
TARGET_JUMLAH = 23000
 
# Jeda antar request (detik) — jangan terlalu cepat agar tidak diblokir
JEDA_MIN = 2
JEDA_MAX = 5

In [3]:
KAMUS_KATEGORI = {
    "Kata Kasar": [
        "anjing", "anjir", "bangsat", "brengsek", "bajingan", "goblok",
        "tolol", "idiot", "bego", "bodoh", "tai", "kampret", "keparat",
        "sialan", "asu", "cok", "kontol", "memek", "pantek", "lonte",
        "sundel", "babi", "monyet", "celeng", "d*mn", "f*ck", "sh*t",
        "a**", "b*tch", "go*blo*", "t*lol", "nj*r", "a*j*ng"
    ],
    "Penipuan": [
        "transfer sekarang", "kirim uang", "mama minta pulsa",
        "menang hadiah", "klik link", "verifikasi akun", "nomor rekening",
        "anda terpilih", "bonus cashback", "undian berhadiah",
        "pinjaman cepat", "tanpa agunan", "bunga rendah", "cair hari ini",
        "investasi menguntungkan", "profit", "passive income", "mlm",
        "wd lancar", "bukti transfer", "jangan bilang siapa",
        "rahasiakan", "bit.ly", "s.id", "tinyurl", "shortlink"
    ],
    "Judi Online": [
        "slot", "gacor", "maxwin", "jackpot", "scatter", "pragmatic",
        "pg soft", "mahjong", "rtp tinggi", "bocoran slot", "deposit",
        "withdraw", "wd", "spin", "bet", "casino", "poker online",
        "togel", "toto", "4d", "3d", "2d", "bandar", "agen resmi",
        "daftar sekarang", "link alternatif", "situs terpercaya"
    ],
    "Pelecehan": [
        "jelek banget", "gemuk", "gendut", "kurus kering", "item",
        "pesek", "pendek", "cebol", "keriting", "bau", "miskin",
        "melarat", "gak laku", "jomblo seumur hidup", "mati aja",
        "bunuh diri", "ga ada gunanya", "sampah masyarakat",
        "memalukan", "aib"
    ],
    "Transaksi": [
        "transfer", "bayar", "pembayaran", "rekening", "bca", "bri",
        "mandiri", "bni", "dana", "ovo", "gopay", "shopeepay",
        "qris", "tagihan", "cicilan", "lunas", "dp", "down payment"
    ],
}
 
# Label berdasarkan kategori
LABEL_MAP = {
    "Kata Kasar": "Berisiko",
    "Penipuan": "Berisiko",
    "Judi Online": "Berisiko",
    "Pelecehan": "Berisiko",
    "Transaksi": "Tidak Berisiko",
    "Netral": "Tidak Berisiko",
}
 
 
# ─────────────────────────────────────────────
# FUNGSI KLASIFIKASI OTOMATIS
# ─────────────────────────────────────────────
 
def klasifikasi_pesan(teks: str) -> tuple[str, str]:
    """
    Mengklasifikasikan teks komentar secara otomatis
    berdasarkan kamus keyword.
    
    Returns:
        (kategori, label) — contoh: ("Kata Kasar", "Berisiko")
    """
    teks_lower = teks.lower()
    
    # Urutan prioritas pengecekan kategori
    urutan_cek = ["Judi Online", "Penipuan", "Kata Kasar", "Pelecehan", "Transaksi"]
    
    for kategori in urutan_cek:
        keywords = KAMUS_KATEGORI.get(kategori, [])
        for kata in keywords:
            # Gunakan regex agar cocok dengan variasi penulisan
            pola = re.compile(re.escape(kata), re.IGNORECASE)
            if pola.search(teks_lower):
                return kategori, LABEL_MAP[kategori]
    
    return "Netral", "Tidak Berisiko"
 

In [4]:
def klasifikasi_pesan(teks: str) -> tuple[str, str]:
    """
    Mengklasifikasikan teks komentar secara otomatis
    berdasarkan kamus keyword.
    
    Returns:
        (kategori, label) — contoh: ("Kata Kasar", "Berisiko")
    """
    teks_lower = teks.lower()
    
    # Urutan prioritas pengecekan kategori
    urutan_cek = ["Judi Online", "Penipuan", "Kata Kasar", "Pelecehan", "Transaksi"]
    
    for kategori in urutan_cek:
        keywords = KAMUS_KATEGORI.get(kategori, [])
        for kata in keywords:
            # Gunakan regex agar cocok dengan variasi penulisan
            pola = re.compile(re.escape(kata), re.IGNORECASE)
            if pola.search(teks_lower):
                return kategori, LABEL_MAP[kategori]
    
    return "Netral", "Tidak Berisiko"

In [5]:
async def scrape_komentar_tiktok(url_video: str, max_komentar: int = 500) -> list[str]:
    """
    Mengambil komentar dari satu video TikTok menggunakan Playwright.
    
    Args:
        url_video: URL video TikTok
        max_komentar: Maksimal komentar yang diambil per video
    
    Returns:
        List berisi teks-teks komentar
    """
    try:
        from playwright.async_api import async_playwright
    except ImportError:
        print("❌ Playwright belum diinstall. Jalankan: pip install playwright && playwright install chromium")
        return []
 
    komentar_list = []
    
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)
            context = await browser.new_context(
                user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
            )
            page = await context.new_page()
            
            print(f"  🌐 Membuka: {url_video}")
            await page.goto(url_video, timeout=30000)
            await page.wait_for_timeout(3000)
            
            # Scroll untuk memuat lebih banyak komentar
            jumlah_scroll = max_komentar // 20
            for i in range(jumlah_scroll):
                await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                await page.wait_for_timeout(random.uniform(1000, 2000))
            
            # Ambil elemen komentar
            elemen_komentar = await page.query_selector_all('[data-e2e="comment-level-1"]')
            
            for elemen in elemen_komentar[:max_komentar]:
                try:
                    teks = await elemen.inner_text()
                    teks_bersih = teks.strip()
                    if teks_bersih and len(teks_bersih) > 3:
                        komentar_list.append(teks_bersih)
                except Exception:
                    continue
            
            await browser.close()
            print(f"  ✅ Berhasil mengambil {len(komentar_list)} komentar")
            
    except Exception as e:
        print(f"  ⚠️ Gagal scraping {url_video}: {e}")
    
    return komentar_list

In [6]:
def simpan_ke_csv(data: list[dict], nama_file: str, mode: str = "w"):
    """
    Menyimpan data ke file CSV dengan kolom: id, pesan, kategori, label
    
    Args:
        data: List of dict dengan key 'pesan', 'kategori', 'label'
        nama_file: Nama file output .csv
        mode: 'w' untuk tulis baru, 'a' untuk append
    """
    fieldnames = ["id", "pesan", "kategori", "label"]
    
    tulis_header = (mode == "w") or (not os.path.exists(nama_file))
    
    with open(nama_file, mode=mode, newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if tulis_header:
            writer.writeheader()
        writer.writerows(data)
 
 
def baca_id_terakhir(nama_file: str) -> int:
    """Membaca ID terakhir dari CSV yang sudah ada (untuk resume)"""
    if not os.path.exists(nama_file):
        return 0
    try:
        with open(nama_file, "r", encoding="utf-8-sig") as f:
            rows = list(csv.DictReader(f))
            if rows:
                return int(rows[-1]["id"])
    except Exception:
        pass
    return 0

In [7]:
async def main():
    print("=" * 60)
    print("  SCRAPING DATASET KOMENTAR TIKTOK")
    print(f"  Target: {TARGET_JUMLAH:,} komentar")
    print(f"  Output: {NAMA_FILE_OUTPUT}")
    print(f"  Waktu mulai: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)
    
    id_counter = baca_id_terakhir(NAMA_FILE_OUTPUT) + 1
    total_terkumpul = id_counter - 1
    mode_file = "a" if total_terkumpul > 0 else "w"
    
    if total_terkumpul > 0:
        print(f"📂 Melanjutkan dari data sebelumnya ({total_terkumpul:,} data)")
    
    for i, url in enumerate(DAFTAR_URL_VIDEO, 1):
        if TARGET_JUMLAH and total_terkumpul >= TARGET_JUMLAH:
            print(f"\n🎯 Target {TARGET_JUMLAH:,} komentar tercapai!")
            break
        
        sisa_target = (TARGET_JUMLAH - total_terkumpul) if TARGET_JUMLAH else 500
        max_per_video = min(500, sisa_target)
        
        print(f"\n[{i}/{len(DAFTAR_URL_VIDEO)}] Memproses video...")
        
        komentar_mentah = await scrape_komentar_tiktok(url, max_per_video)
        
        if not komentar_mentah:
            print("  ⏩ Lewati video ini (tidak ada komentar)")
            continue
        
        # Klasifikasi dan format data
        batch_data = []
        for teks in komentar_mentah:
            kategori, label = klasifikasi_pesan(teks)
            batch_data.append({
                "id": id_counter,
                "pesan": teks,
                "kategori": kategori,
                "label": label
            })
            id_counter += 1
        
        # Simpan batch ke CSV
        simpan_ke_csv(batch_data, NAMA_FILE_OUTPUT, mode=mode_file)
        mode_file = "a"  # Selanjutnya selalu append
        
        total_terkumpul += len(batch_data)
        
        # Statistik per batch
        berisiko = sum(1 for d in batch_data if d["label"] == "Berisiko")
        aman = len(batch_data) - berisiko
        
        print(f"  📊 Batch ini: {len(batch_data)} komentar | Berisiko: {berisiko} | Aman: {aman}")
        print(f"  📈 Total terkumpul: {total_terkumpul:,} / {TARGET_JUMLAH:,}")
        
        # Jeda agar tidak diblokir
        if i < len(DAFTAR_URL_VIDEO):
            jeda = random.uniform(JEDA_MIN, JEDA_MAX)
            print(f"  ⏳ Jeda {jeda:.1f} detik...")
            await asyncio.sleep(jeda)
    
    # Ringkasan akhir
    print("\n" + "=" * 60)
    print("  SELESAI! RINGKASAN DATASET")
    print("=" * 60)
    
    if os.path.exists(NAMA_FILE_OUTPUT):
        with open(NAMA_FILE_OUTPUT, "r", encoding="utf-8-sig") as f:
            semua_data = list(csv.DictReader(f))
        
        total = len(semua_data)
        per_kategori = {}
        per_label = {}
        
        for row in semua_data:
            kat = row["kategori"]
            lab = row["label"]
            per_kategori[kat] = per_kategori.get(kat, 0) + 1
            per_label[lab] = per_label.get(lab, 0) + 1
        
        print(f"  Total data    : {total:,} baris")
        print(f"\n  Distribusi Label:")
        for lab, jml in per_label.items():
            persen = (jml / total) * 100
            print(f"    {lab:<20}: {jml:>6,} ({persen:.1f}%)")
        print(f"\n  Distribusi Kategori:")
        for kat, jml in sorted(per_kategori.items(), key=lambda x: -x[1]):
            persen = (jml / total) * 100
            print(f"    {kat:<20}: {jml:>6,} ({persen:.1f}%)")
        
        print(f"\n  ✅ File tersimpan: {NAMA_FILE_OUTPUT}")
    
    print("=" * 60)

In [8]:
def demo_klasifikasi():
    """
    Uji coba klasifikasi dengan data contoh tanpa perlu internet.
    Jalankan: python scraping_tiktok_dataset.py --demo
    """
    contoh_data = [
        "dasar sialan lu, gak berguna!",
        "apa-apa lo ribut, goblok banget",
        "Transfer BCA 50rb ya kak buat bayar",
        "Daftar slot gacor maxwin disini, WD lancar!",
        "klik bit.ly/hadiahgratis menang 10 juta sekarang",
        "wah lucu banget videonya hahaha",
        "setuju banget sama pendapatnya",
        "mama minta pulsa dong anakku",
        "jelek banget mukanya, gendut lagi",
        "situs terpercaya pragmatic scatter hitam daftar sekarang",
    ]
    
    print("\n🧪 DEMO KLASIFIKASI OTOMATIS")
    print("-" * 60)
    print(f"{'ID':<4} {'PESAN':<40} {'KATEGORI':<15} {'LABEL'}")
    print("-" * 60)
    
    hasil = []
    for i, teks in enumerate(contoh_data, 1):
        kategori, label = klasifikasi_pesan(teks)
        singkat = teks[:37] + "..." if len(teks) > 37 else teks
        print(f"{i:<4} {singkat:<40} {kategori:<15} {label}")
        hasil.append({"id": i, "pesan": teks, "kategori": kategori, "label": label})
    
    # Simpan hasil demo
    simpan_ke_csv(hasil, "demo_output.csv", mode="w")
    print(f"\n✅ Hasil demo disimpan ke: demo_output.csv")

In [10]:
if __name__ == "__main__":
    import sys
    
    if "--demo" in sys.argv:
        # Mode demo: test klasifikasi tanpa internet
        demo_klasifikasi()
    else:
        # Mode scraping penuh
        if not DAFTAR_URL_VIDEO or DAFTAR_URL_VIDEO[0].endswith("1234567890"):
            print("⚠️  PERHATIAN: Kamu belum mengisi URL video TikTok!")
            print("    Edit variabel DAFTAR_URL_VIDEO di bagian atas script ini.")
            print("\n    Untuk test klasifikasi tanpa internet, jalankan:")
            print("    python scraping_tiktok_dataset.py --demo")
        else:
            await main()

  SCRAPING DATASET KOMENTAR TIKTOK
  Target: 23,000 komentar
  Output: dataset_tiktok.csv
  Waktu mulai: 2026-05-24 16:42:31

[1/8] Memproses video...
  ⚠️ Gagal scraping https://vt.tiktok.com/ZSxmNoQ4Q/: 
  ⏩ Lewati video ini (tidak ada komentar)

[2/8] Memproses video...
  ⚠️ Gagal scraping https://vt.tiktok.com/ZSxmF9oqJ/: 
  ⏩ Lewati video ini (tidak ada komentar)

[3/8] Memproses video...
  ⚠️ Gagal scraping https://vt.tiktok.com/ZSxmFVNY4/: 
  ⏩ Lewati video ini (tidak ada komentar)

[4/8] Memproses video...
  ⚠️ Gagal scraping https://vt.tiktok.com/ZSxmFTtAu/: 
  ⏩ Lewati video ini (tidak ada komentar)

[5/8] Memproses video...
  ⚠️ Gagal scraping https://vt.tiktok.com/ZSxmYkrqy/: 
  ⏩ Lewati video ini (tidak ada komentar)

[6/8] Memproses video...
  ⚠️ Gagal scraping https://vt.tiktok.com/ZSxm2QSkk/: 
  ⏩ Lewati video ini (tidak ada komentar)

[7/8] Memproses video...
  ⚠️ Gagal scraping https://vt.tiktok.com/ZSxmj5QyF/: 
  ⏩ Lewati video ini (tidak ada komentar)

[8/8] Mempros